[spambase dataset exercise](https://github.com/rosarioscavo/UCI-Spambase/)

In [ ]:
# Data
from ucimlrepo import fetch_ucirepo
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score, cross_val_predict
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# Fetch dataset
spambase = fetch_ucirepo(id=94)

# Data (as pandas dataframes)
X = spambase.data.features
y = spambase.data.targets["Class"]

# Clean column names (replace invalid characters)
X.columns = [
    str(col).replace('[', '_').replace(']', '_').replace('<', '_').replace('>', '_')
    for col in X.columns
]

# ── Cross-Validation Setup ──────────────────────────────────────────────────
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

model = XGBClassifier(
    n_estimators=300,
    learning_rate=0.1,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric='logloss',
    random_state=42
)

# ── CV Accuracy Scores ──────────────────────────────────────────────────────
cv_scores = cross_val_score(model, X, y, cv=cv, scoring='accuracy', n_jobs=-1)

print("── Cross-Validation Accuracy Scores ──")
for i, score in enumerate(cv_scores, 1):
    print(f"  Fold {i}: {score:.4f}")
print(f"\n  Mean Accuracy : {cv_scores.mean():.4f}")
print(f"  Std Deviation : {cv_scores.std():.4f}")

# ── Out-of-Fold Predictions (for confusion matrix & report) ────────────────
y_pred_oof = cross_val_predict(model, X, y, cv=cv, n_jobs=-1)

print("\n── Classification Report (Out-of-Fold) ──")
print(classification_report(y, y_pred_oof))

# ── Confusion Matrix ────────────────────────────────────────────────────────
cm = confusion_matrix(y, y_pred_oof)
plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('Confusion Matrix (Out-of-Fold CV Predictions)')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.tight_layout()
plt.show()

# ── CV Score Distribution ───────────────────────────────────────────────────
plt.figure(figsize=(7, 4))
plt.bar(range(1, len(cv_scores) + 1), cv_scores, color='steelblue', alpha=0.8)
plt.axhline(cv_scores.mean(), color='red', linestyle='--', label=f'Mean: {cv_scores.mean():.4f}')
plt.xlabel('Fold')
plt.ylabel('Accuracy')
plt.title('CV Accuracy per Fold')
plt.ylim(0.9, 1.0)
plt.legend()
plt.tight_layout()
plt.show()

# ── Feature Importances (refit on full data) ────────────────────────────────
model.fit(X, y)
importance = model.feature_importances_
indices = np.argsort(importance)[::-1]

plt.figure(figsize=(10, 6))
plt.bar(range(20), importance[indices][:20], color='steelblue', alpha=0.8)
plt.xticks(range(20), X.columns[indices][:20], rotation=90)
plt.title('Top 20 Feature Importances (Full Data Fit)')
plt.tight_layout()
plt.show()